# Early-Stage Pancreatic Cancer Detection — Random Forest (Colab run)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kareem1207/Early-Stage-Pancreatic-Cancer-Detection-Using-Random-Forest-Classification/blob/main/notebooks/colab_run.ipynb)

Runs the whole pipeline from the GitHub repo: **train** (SMOTE + Random Forest + GridSearchCV) → **evaluate** (metrics, confusion matrix, ROC, feature importance) → **predict** for a custom patient.

**How to use:** menu *Runtime → Run all* (about 1–2 minutes on the free CPU runtime; no GPU needed).

Colab ships newer library versions than the report was produced with, so a number can differ from the report in the last decimal.

## 1. Get the code and install the one missing package

In [ ]:
import os

REPO = "Early-Stage-Pancreatic-Cancer-Detection-Using-Random-Forest-Classification"
if not os.path.exists(REPO):
    !git clone -q -b main https://github.com/kareem1207/{REPO}.git
%cd {REPO}/src

# Colab already has pandas, scikit-learn, matplotlib, seaborn and joblib.
# Do NOT run `pip install -r requirements.txt` here: it pins older numpy/scikit-learn and can break the runtime.
%pip install -q imbalanced-learn

## 2. (Optional) Use your own dataset

By default the notebook uses the dataset bundled in the repo (`data/raw/urinary_biomarkers.csv`).
To run on a different CSV (for example the real Kaggle file), set `UPLOAD_REAL_DATA = True`, run the cell, and pick the file when asked.
It must have the same columns as the bundled one: `sample_id, patient_cohort, sample_origin, age, sex, diagnosis, stage, benign_sample_diagnosis, plasma_CA19_9, creatinine, LYVE1, REG1B, TFF1, REG1A`.

In [ ]:
UPLOAD_REAL_DATA = False

if UPLOAD_REAL_DATA:
    from google.colab import files
    uploaded = files.upload()                       # choose the CSV
    name = next(iter(uploaded))
    os.replace(name, "../data/raw/urinary_biomarkers.csv")
    print("Using uploaded dataset:", name)
else:
    print("Using the dataset bundled in the repo.")

## 3. Train — preprocessing, SMOTE, Random Forest, GridSearchCV (270 fits)

In [ ]:
%run train.py

## 4. Evaluate on the held-out test set

In [ ]:
%run evaluate.py

In [ ]:
from IPython.display import Image, display

for name in ["confusion_matrix", "roc_curve", "feature_importance"]:
    display(Image(f"../reports/{name}.png", width=520))

## 5. Predict for a custom patient

Edit the values and re-run the cell.

In [ ]:
import pandas as pd
from predict import predict

patient = pd.DataFrame([{
    "age": 68, "sex": "F", "patient_cohort": "Cohort1", "sample_origin": "BPTB",
    "plasma_CA19_9": 145.0, "creatinine": 1.2,
    "LYVE1": 6.8, "REG1B": 38.5, "TFF1": 120.0, "REG1A": 22.0,
}])

result = predict(patient)
result[["predicted_diagnosis", "probability_Control", "probability_Benign", "probability_PDAC"]].round(3)